In [1]:
import sys
sys.path.append("../")

import pandas as pd
from IPython.display import Markdown, display

from evaluation.reporting import export_class_routing_report_to_markdown
from prompts.routing_prompts import routing_prompt
from models.routing import router, RouteDecision

CONFIDENCE_THRESHOLD = 0.65



In [2]:
# get model for classification
# get prompt for classification
# augment query with prompt
# generate output

In [3]:
def generate(query: str) -> RouteDecision:

    try:
        return router.complete(query).raw

    except Exception as e:
        
        print(f"Routing failed: {e}")

        return RouteDecision(
            route="faq",
            confidence=0.0,
        )

In [4]:
def make_prompt(query) -> str:
    return routing_prompt.render(query=query)

In [5]:
def apply_confidence_threshold(response: RouteDecision, threshold):
    if response.confidence < threshold:
        return None, response.confidence

    return response.route, response.confidence

In [6]:
def process_and_print_query(query: str, correct_label: str, route, confidence) -> dict:
    is_correct = route == correct_label

    result = {
        "query": query,
        "expected": correct_label,
        "predicted": route,
        "confidence": round(confidence, 4) if confidence is not None else None,
        "is_correct": is_correct,
    }

    print(f"Query: {query}")
    print(f"Expected: {correct_label}")
    print(f"Predicted: {route}")

    if confidence is not None:
        print(f"Confidence: {confidence:.2f}")

    print(f"Correct: {'yes' if is_correct else 'no'}")
    print("-" * 60)
    return result



In [7]:
def get_class_query(query) -> str:
    prompt = make_prompt(query)
    response = generate(prompt)
    route, confidence = apply_confidence_threshold(response, CONFIDENCE_THRESHOLD)
    return route, confidence


In [8]:
queries = [
    'How old are you?',
    'Tell me a joke about online shopping.',
    'Translate \"blue shirt\" into Polish.',
    'What is the weather in Warsaw today?',
    'Solve 17 * 23 for me.',
    'Who won the football World Cup in 2018?',
    'Summarize Dune in two sentences.',
    'What is your return policy?', 
    'How can I reset my password?',
    'Can I change the email address connected to my account?',
    'How do I track my parcel after it is shipped?',
    'What should I do if my package arrived damaged?',
    'How long does it usually take to get a refund?',
    'How can I download an invoice for my order?',
    'Give me three examples of blue T-shirts you have available.', 
    'How can I contact the user support?', 
    'Do you have blue Dresses?',
    'Show me black ankle boots in size 39.',
    'Find a beige trench coat for spring.',
    'I need a leather handbag for work meetings.',
    'Show me white sneakers for men.',
    'Recommend accessories for a navy evening gown.',
    'Create a look suitable for a wedding party happening during dawn.'
]



In [9]:
labels = [
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    'faq',
    'faq',
    'faq',
    'faq',
    'faq',
    'faq',
    'faq',
    'product',
    'faq',
    'product',
    'product',
    'product',
    'product',
    'product',
    'product',
    'product'
]



In [10]:
def build_markdown_report(results_df: pd.DataFrame, output_path: str = "../reports/class_routing_evaluation.md"):
    return export_class_routing_report_to_markdown(
        results_df,
        output_path,
        notebook_path="notebooks/06_class_routing.ipynb",
    )


results = []

for query, correct_label in zip(queries, labels):
    route, confidence = get_class_query(query)
    results.append(process_and_print_query(query, correct_label, route, confidence))

results_df = pd.DataFrame(results)
report_path, summary_df, class_metrics_df, confusion_df, per_case_df = build_markdown_report(results_df)

display(Markdown(f"Markdown report saved to: `{report_path}`"))



Query: How old are you?
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Tell me a joke about online shopping.
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Translate "blue shirt" into Polish.
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: What is the weather in Warsaw today?
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Solve 17 * 23 for me.
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Who won the football World Cup in 2018?
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Summarize Dune in two sentenc

Markdown report saved to: `../reports/class_routing_evaluation.md`